# ✨ Técnicas de Procesamiento de Lenguaje Natural (PLN) 📖  
## 📝 Cuaderno 03 - *Modelos de lenguaje probabilísticos*  
### *Cuaderno previo -> Preprocesamiento de textos (parte 2)*

## 🏆 Clase: Modelos de lenguaje, Estimación estadística de LM, Evaluación de LM, herramientas

📅 **Desarrollado por:** [Julian David Echeverry](mailto:jde@utp.edu.co)

---

## 🔹 Sección 0: Instalación de paquetes y carga de librerías
🔍 **Objetivo:** En esta sección se instalarán los paquetes y se cargarán las librerías necesarias para ejecutar las celdas posteriores

In [20]:
# Instalación y descarga de recursos
!pip install nltk spacy
!pip install nltk markovify
# Descargar modelos y recursos
import re
import math
import nltk
import markovify
import random
from nltk import ngrams, bigrams, FreqDist, word_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from collections import defaultdict, Counter


import spacy
try:
    nlp_en = spacy.load("en_core_web_sm")
except:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp_en = spacy.load("en_core_web_sm")

try:
    nlp_es = spacy.load("es_core_news_sm")
except:
    from spacy.cli import download
    download("es_core_news_sm")
    nlp_es = spacy.load("es_core_news_sm")

print("Recursos cargados correctamente.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Recursos cargados correctamente.


---
## 🔹 Sección 1: Modelos Probabilísticos de Lenguaje

#### 🔍 **Objetivo**  
Explorar los fundamentos y aplicaciones de los modelos probabilísticos de lenguaje, implementando ejemplos prácticos en inglés y español. Se estudiarán conceptos como la regla de la cadena, la hipótesis de Markov, modelos n-grama y técnicas de suavizado, además de evaluar los modelos mediante la métrica de perplejidad.

---

#### 📌 **Introducción**

Los modelos probabilísticos asignan probabilidades a secuencias de palabras y son fundamentales en tareas de procesamiento del lenguaje natural (PLN) como la predicción de texto, reconocimiento de voz y traducción automática. En este cuaderno se presentarán:

- **Fundamentos teóricos:** desde la regla de la cadena y la hipótesis de Markov hasta la construcción de modelos n-grama.
- **Técnicas de Suavizado:** como Laplace para evitar el problema de la dispersión de datos.
- **Evaluación de Modelos:** mediante el cálculo de la perplejidad.
- **Ejemplos Prácticos:** en inglés y español usando librerías como **NLTK** y **spaCy**.

---

#### 📚 **Estructura del Cuaderno**

1. **Fundamentos Teóricos y Preparación**
   - Introducción a los conceptos probabilísticos y la hipótesis de Markov.
2. **Ejemplo Práctico con NLTK**
   - Construcción de un modelo bigrama (n-grama con n=2) en inglés y español.
   - Aplicación de suavizado Laplace.
3. **Ejemplo Práctico con spaCy**
   - Tokenización y construcción de un modelo bigrama usando spaCy.
4. **Evaluación del Modelo: Cálculo de Perplejidad**
   - Función para medir la perplejidad del modelo.
5. **Conclusiones y Debate**
   - Resumen de lo aprendido y discusión de ventajas/desventajas.

---

### Modelo Bigrama con NLTK (Ejemplo en Inglés y Español)
En este ejemplo se construye un modelo bigrama a partir de textos en inglés y español, aplicando suavizado Laplace para estimar las probabilidades.

In [19]:
# Función para limpiar y tokenizar el texto
def tokenize_text(text, language='english'):
    tokens = word_tokenize(text)
    # Filtrar tokens alfanuméricos y pasar a minúsculas
    tokens = [token.lower() for token in tokens if re.match(r'\w+', token)]
    # Eliminar stop words
    if language == 'english':
        stops = set(stopwords.words('english'))
    else:
        stops = set(stopwords.words('spanish'))
    tokens = [token for token in tokens if token not in stops]
    return tokens

# Textos de ejemplo
text_en = ("New York is a city that never sleeps. The city is vibrant and full of opportunities. "
           "New York offers endless experiences in art, culture, and nightlife.")

text_es = ("Bogotá es una ciudad llena de contrastes, donde la modernidad convive con la historia. "
           "Bogotá, la capital colombiana ofrece una rica cultura y diversidad en cada uno de sus barrios.")

# Tokenización
tokens_en = tokenize_text(text_en, language='english')
tokens_es = tokenize_text(text_es, language='spanish')

# Construcción de bigramas
bigrams_en = list(bigrams(tokens_en))
bigrams_es = list(bigrams(tokens_es))

print("---------------------------------------------")
print("---------Bigramas en español---")
for item in bigrams_es:
    print(item)
print("---------------------------------------------")

print("---------Bigramas en inglés---")
for item in bigrams_en:
    print(item)
print("---------------------------------------------")

# Frecuencias de unigramas y bigramas
freq_uni_en = FreqDist(tokens_en)
freq_bi_en = FreqDist(bigrams_en)
freq_uni_es = FreqDist(tokens_es)
freq_bi_es = FreqDist(bigrams_es)

# Tamaño del vocabulario
V_en = len(set(tokens_en))
V_es = len(set(tokens_es))

# Función para calcular probabilidades SIN suavizado Laplace
def ngram_prob(bigram, freq_uni, freq_bi):
    previous = bigram[0]
    return (freq_bi[bigram]) / (freq_uni[previous])

# Función para calcular probabilidades con suavizado Laplace
def laplace_smoothed_prob(bigram, freq_uni, freq_bi, V):
    previous = bigram[0]
    return (freq_bi[bigram] + 1) / (freq_uni[previous] + V)

# Calcular probabilidades de algunos bigramas en español (SIN SUAVIZADO)
ngrams_probs_es = {bg: ngram_prob(bg, freq_uni_es, freq_bi_es) for bg in freq_bi_es}
print("---------------------------------------------")
print("---------Bigramas y probabilidades (SIN suavizado) - Español:")
for bg, prob in ngrams_probs_es.items():
    print(f"{bg}: {prob:.4f}")
print("---------------------------------------------")

# Calcular probabilidades de algunos bigramas en inglés (SIN SUAVIZADO)
ngrams_probs_en = {bg: ngram_prob(bg, freq_uni_en, freq_bi_en) for bg in freq_bi_en}
print("---------------------------------------------")
print("---------Bigramas y probabilidades (SIN suavizado) - Inglés:")
for bg, prob in ngrams_probs_en.items():
    print(f"{bg}: {prob:.4f}")
print("---------------------------------------------")

# Calcular probabilidades de algunos bigramas en inglés
laplace_probs_en = {bg: laplace_smoothed_prob(bg, freq_uni_en, freq_bi_en, V_en) for bg in freq_bi_en}
print("---------------------------------------------")
print("---------Bigramas y probabilidades (suavizado Laplace) - Inglés:")
for bg, prob in laplace_probs_en.items():
    print(f"{bg}: {prob:.4f}")
print("---------------------------------------------")

# Calcular probabilidades de algunos bigramas en español
laplace_probs_es = {bg: laplace_smoothed_prob(bg, freq_uni_es, freq_bi_es, V_es) for bg in freq_bi_es}
print("---------------------------------------------")
print("---------Bigramas y probabilidades (suavizado Laplace) - Español:")
for bg, prob in laplace_probs_es.items():
    print(f"{bg}: {prob:.4f}")
print("---------------------------------------------")

---------------------------------------------
---------Bigramas en español---
('bogotá', 'ciudad')
('ciudad', 'llena')
('llena', 'contrastes')
('contrastes', 'modernidad')
('modernidad', 'convive')
('convive', 'historia')
('historia', 'bogotá')
('bogotá', 'capital')
('capital', 'colombiana')
('colombiana', 'ofrece')
('ofrece', 'rica')
('rica', 'cultura')
('cultura', 'diversidad')
('diversidad', 'cada')
('cada', 'barrios')
---------------------------------------------
---------Bigramas en inglés---
('new', 'york')
('york', 'city')
('city', 'never')
('never', 'sleeps')
('sleeps', 'city')
('city', 'vibrant')
('vibrant', 'full')
('full', 'opportunities')
('opportunities', 'new')
('new', 'york')
('york', 'offers')
('offers', 'endless')
('endless', 'experiences')
('experiences', 'art')
('art', 'culture')
('culture', 'nightlife')
---------------------------------------------
---------------------------------------------
---------Bigramas y probabilidades (SIN suavizado) - Español:
('bogotá', 

### Modelo Bigrama con spaCy
En este ejemplo se utiliza spaCy para tokenizar y construir un modelo bigrama, lo que permite comparar la tokenización y extracción de bigramas obtenida con otra herramienta.

In [23]:
# Función para tokenizar usando spaCy
def spacy_tokenize(text, nlp):
    doc = nlp(text)
    # Filtrar tokens alfabéticos y pasar a minúsculas
    tokens = [token.text.lower() for token in doc if token.is_alpha]
    return tokens

# Tokenización con spaCy
tokens_spacy_en = spacy_tokenize(text_en, nlp_en)
tokens_spacy_es = spacy_tokenize(text_es, nlp_es)

# Construcción de bigramas con spaCy
bigrams_spacy_en = list(bigrams(tokens_spacy_en))
bigrams_spacy_es = list(bigrams(tokens_spacy_es))

# Frecuencias de unigramas y bigramas
freq_uni_spacy_en = FreqDist(tokens_spacy_en)
freq_bi_spacy_en = FreqDist(bigrams_spacy_en)
freq_uni_spacy_es = FreqDist(tokens_spacy_es)
freq_bi_spacy_es = FreqDist(bigrams_spacy_es)

# Tamaño del vocabulario
V_spacy_en = len(set(tokens_spacy_en))
V_spacy_es = len(set(tokens_spacy_es))

# Calcular probabilidades con suavizado Laplace para spaCy (Ejemplo en Inglés)
laplace_probs_spacy_en = {bg: laplace_smoothed_prob(bg, freq_uni_spacy_en, freq_bi_spacy_en, V_spacy_en) for bg in freq_bi_spacy_en}
print("\n---------------------------------------------")
print("------Bigramas y probabilidades (suavizado Laplace) con spaCy - Inglés:")
for bg, prob in laplace_probs_spacy_en.items():
    print(f"{bg}: {prob:.4f}")
print("---------------------------------------------")

# Calcular probabilidades con suavizado Laplace para spaCy (Ejemplo en Español)
laplace_probs_spacy_es = {bg: laplace_smoothed_prob(bg, freq_uni_spacy_es, freq_bi_spacy_es, V_spacy_es) for bg in freq_bi_spacy_es}
print("------Bigramas y probabilidades (suavizado Laplace) con spaCy - Español:")
for bg, prob in laplace_probs_spacy_es.items():
    print(f"{bg}: {prob:.4f}")
print("---------------------------------------------")


---------------------------------------------
------Bigramas y probabilidades (suavizado Laplace) con spaCy - Inglés:
('new', 'york'): 0.1304
('york', 'is'): 0.0870
('is', 'a'): 0.0870
('a', 'city'): 0.0909
('city', 'that'): 0.0870
('that', 'never'): 0.0909
('never', 'sleeps'): 0.0909
('sleeps', 'the'): 0.0909
('the', 'city'): 0.0909
('city', 'is'): 0.0870
('is', 'vibrant'): 0.0870
('vibrant', 'and'): 0.0909
('and', 'full'): 0.0870
('full', 'of'): 0.0909
('of', 'opportunities'): 0.0909
('opportunities', 'new'): 0.0909
('york', 'offers'): 0.0870
('offers', 'endless'): 0.0909
('endless', 'experiences'): 0.0909
('experiences', 'in'): 0.0909
('in', 'art'): 0.0909
('art', 'culture'): 0.0909
('culture', 'and'): 0.0909
('and', 'nightlife'): 0.0870
---------------------------------------------
------Bigramas y probabilidades (suavizado Laplace) con spaCy - Español:
('bogotá', 'es'): 0.0741
('es', 'una'): 0.0769
('una', 'ciudad'): 0.0741
('ciudad', 'llena'): 0.0769
('llena', 'de'): 0.0769
('de

---

## 🔹 Sección 2: generación de textos a partir de LM entrenados

### Corpus
Partimos de dos textos de entrenamiento (uno en español y otro en inglés) de varios párrafos. Cada uno tiene un estilo natural y trata temas similares (tecnología, conexión, educación) para que el modelo tenga suficiente variedad de palabras y estructuras.

### `markovify.Text`
Creamos un modelo para cada idioma con state_size=2 (bigramas).

### Generación
Usamos un bucle para generar 3 oraciones por idioma. Se añadió una verificación con if porque, con corpus pequeños, make_sentence() a veces devuelve None si no puede formar una oración coherente.

### Salida
Se imprimen las oraciones generadas con etiquetas claras para distinguir entre español e inglés.

### Resultados esperados
Debido a la naturaleza aleatoria de las cadenas de Markov, las oraciones generadas variarán en cada ejecución.

---

### 🔥 Presten atención a esta sección (vamos a ver que el código NO funciona bien y vamos a entender qué pasa y cómo mejorarlo)

Vamos a crear corpus (tanto para español como para inglés).

In [28]:
# Corpus en español
corpus_es = """
La tecnología ha revolucionado la forma en que vivimos. Cada día surgen nuevas herramientas que facilitan nuestras tareas diarias.
Las personas están más conectadas que nunca gracias a internet. Sin embargo, también hay desafíos: la privacidad es un tema candente.
Algunos dicen que el futuro será dominado por la inteligencia artificial, mientras que otros creen que el ser humano siempre tendrá el control.
La educación también se transforma con cursos en línea y plataformas digitales. Todo esto nos lleva a reflexionar sobre cómo usamos estas herramientas.
El mundo avanza rápido, y adaptarse es esencial para no quedarse atrás.
"""

# Corpus en inglés
corpus_en = """
Technology has changed the way we live in incredible ways. Every day, new tools emerge to make our lives easier and more efficient.
People are more connected than ever before, thanks to the internet and social media. However, there are challenges too: privacy concerns are growing.
Some believe artificial intelligence will shape the future, while others argue that humans will always remain in charge.
Education is evolving with online courses and digital platforms, opening new opportunities. This makes us think about how we use these advancements.
The world is moving fast, and staying adaptable is key to keeping up.
"""

### ✏️ Con esos corpus creamos modelos de bigrama y a partir de esos modelos generamos textos aleatorios.

In [29]:
# Debimos haber importado <import markovify> desde antes

# Construir modelos de Markov
modelo_es = markovify.Text(corpus_es, state_size=2)  # Modelo en español con bigramas
modelo_en = markovify.Text(corpus_en, state_size=2)  # Modelo en inglés con bigramas

# Generar 3 oraciones en español
print("=== Oraciones generadas en español ===")
for _ in range(3):
    oracion_es = modelo_es.make_sentence()
    if oracion_es:  # Verifica que se haya generado una oración
        print(f"Texto generado: {oracion_es}")
    else:
        print("No se pudo generar una oración válida.")

# Generar 3 oraciones en inglés
print("\n=== Orations generated in English ===")
for _ in range(3):
    oration_en = modelo_en.make_sentence()
    if oration_en:  # Verifica que se haya generado una oración
        print(f"Generated text: {oration_en}")
    else:
        print("Could not generate a valid sentence.")

=== Oraciones generadas en español ===
No se pudo generar una oración válida.
No se pudo generar una oración válida.
No se pudo generar una oración válida.

=== Orations generated in English ===
Could not generate a valid sentence.
Could not generate a valid sentence.
Could not generate a valid sentence.


### ☝️⬆️ Si revisamos la salida previa, vemos que no hizo nada.

**¿Qué pasó?**

- Corpus pequeño: Aunque el texto tiene varios párrafos, sigue siendo limitado para un modelo de Markov con state_size=2. Necesita más datos para encontrar patrones consistentes.
- Falta de finales claros: Si el corpus no tiene suficientes puntos (.) o estructuras de oración completas, el modelo no sabe cómo terminar las oraciones.
- Restricciones internas: markovify tiene un límite predeterminado de intentos para generar una oración; si falla demasiadas veces, devuelve None.

---


Vamos a mejorar la estimación.

In [32]:
############ Pasos para mejorar la estimación ############

# 1. Le vamos a pasamos corpus más extensos

# Corpus en español (más extenso)
corpus_es = """
La tecnología ha revolucionado la forma en que vivimos nuestras vidas cotidianas. Cada día surgen nuevas herramientas que facilitan nuestras tareas diarias y nos hacen más productivos.
Las personas están más conectadas que nunca gracias a internet y a las redes sociales como Facebook, Twitter o Instagram. Sin embargo, también hay desafíos importantes que enfrentar.
La privacidad es un tema candente en la era digital, y mucha gente teme que sus datos sean utilizados sin permiso. Algunos expertos dicen que el futuro será dominado por la inteligencia artificial y las máquinas avanzadas.
Otros creen que el ser humano siempre tendrá el control y que la tecnología solo será una herramienta en nuestras manos.
La educación también se transforma con cursos en línea, plataformas digitales y nuevas formas de aprendizaje a distancia. Todo esto nos lleva a reflexionar sobre cómo usamos estas herramientas en nuestra vida diaria.
El mundo avanza rápido, y adaptarse es esencial para no quedarse atrás en este entorno cambiante. Por ejemplo, las empresas ahora dependen de la tecnología para competir en el mercado global.
Además, la inteligencia artificial está siendo utilizada en campos como la medicina, la ingeniería y el entretenimiento. Aunque hay beneficios, también hay riesgos que debemos considerar con cuidado.
El futuro es incierto, pero emocionante al mismo tiempo. ¿Cómo será el mundo en 20 años? Nadie lo sabe con certeza, pero la tecnología seguirá siendo clave.
"""

# Corpus en inglés (más extenso)
corpus_en = """
Technology has changed the way we live in incredible and unexpected ways. Every day, new tools emerge to make our lives easier, more efficient, and more enjoyable.
People are more connected than ever before, thanks to the internet and social media platforms like Facebook, Twitter, and Instagram. However, there are also significant challenges to face.
Privacy concerns are growing in the digital age, and many people worry about their personal data being misused. Some experts believe artificial intelligence and advanced machines will shape the future of humanity.
Others argue that humans will always remain in charge and that technology will stay as a tool in our hands. Education is evolving with online courses, digital platforms, and new methods of remote learning.
This makes us think about how we use these advancements in our daily lives. The world is moving fast, and staying adaptable is key to keeping up with this changing environment.
For instance, companies now rely on technology to compete in the global market and stay relevant. Moreover, artificial intelligence is being applied in fields like medicine, engineering, and entertainment.
While there are clear benefits, there are also risks that we must consider carefully. The future is uncertain but exciting at the same time. What will the world look like in 20 years? No one knows for sure, but technology will undoubtedly play a central role.
"""

# Construir modelos de Markov
modelo_es = markovify.Text(corpus_es, state_size=2)
modelo_en = markovify.Text(corpus_en, state_size=2)

# Generar 5 oraciones en español con más intentos
print("=== Oraciones generadas en español ===")
for _ in range(5):
    oracion_es = modelo_es.make_sentence(tries=100)  # Aumentamos los intentos a 100
    if oracion_es:
        print(f"Texto generado: {oracion_es}")
    else:
        print("No se pudo generar una oración válida (intenta con un corpus más grande).")

# Generar 5 oraciones en inglés con más intentos
print("\n=== Orations generated in English ===")
for _ in range(5):
    oration_en = modelo_en.make_sentence(tries=100)  # Aumentamos los intentos a 100
    if oration_en:
        print(f"Generated text: {oration_en}")
    else:
        print("Could not generate a valid sentence (try a larger corpus).")

=== Oraciones generadas en español ===
Texto generado: Aunque hay beneficios, también hay desafíos importantes que enfrentar.
Texto generado: Nadie lo sabe con certeza, pero la tecnología solo será una herramienta en nuestras manos.
Texto generado: Por ejemplo, las empresas ahora dependen de la tecnología solo será una herramienta en nuestras manos.
Texto generado: Aunque hay beneficios, también hay desafíos importantes que enfrentar.
Texto generado: Aunque hay beneficios, también hay desafíos importantes que enfrentar.

=== Orations generated in English ===
Generated text: For instance, companies now rely on technology to compete in the digital age, and many people worry about their personal data being misused.
Generated text: Others argue that humans will always remain in charge and that technology will undoubtedly play a central role.
Generated text: No one knows for sure, but technology will stay as a tool in our daily lives.
Generated text: Privacy concerns are growing in the glob

### ☝️⬆️ Si revisamos la salida previa, vemos que la estimación mejoró, sin ser perfecta, pero mejoró.

**¿Qué pasó?**
- Mejoramos el tamaño del corpus.
- Aumentamos el número de intentos para la generación.

---

Probemos ahora con un corpus aún más extenso y con algunas restricciones adicionales para ver si eso mejora la estimación.

Algunas mejoras posibles:

- Aumentar aún más el corpus: Más texto significa más combinaciones posibles.
- Reducir state_size a 1: Usar unigramas para más flexibilidad (aunque puede ser menos coherente).
- Combinar modelos: Usar múltiples versiones del corpus para mezclar patrones.
- Ajustar parámetros: Usar `make_short_sentence` o aumentar `tries` con más aleatoriedad.

In [43]:
# Corpus en español (aún más extenso y variado)
corpus_es = """
La tecnología ha revolucionado la forma en que vivimos nuestras vidas cotidianas. Cada día surgen nuevas herramientas que facilitan nuestras tareas diarias y nos hacen más productivos.
Las personas están más conectadas que nunca gracias a internet y a las redes sociales como Facebook, Twitter o Instagram. Sin embargo, también hay desafíos importantes que enfrentar.
La privacidad es un tema candente en la era digital, y mucha gente teme que sus datos sean utilizados sin permiso. Algunos expertos dicen que el futuro será dominado por la inteligencia artificial y las máquinas avanzadas.
Otros creen que el ser humano siempre tendrá el control y que la tecnología solo será una herramienta en nuestras manos. La educación también se transforma con cursos en línea, plataformas digitales y nuevas formas de aprendizaje a distancia.
Todo esto nos lleva a reflexionar sobre cómo usamos estas herramientas en nuestra vida diaria. El mundo avanza rápido, y adaptarse es esencial para no quedarse atrás en este entorno cambiante.
Por ejemplo, las empresas ahora dependen de la tecnología para competir en el mercado global. Además, la inteligencia artificial está siendo utilizada en campos como la medicina, la ingeniería y el entretenimiento.
Aunque hay beneficios, también hay riesgos que debemos considerar con cuidado. El futuro es incierto, pero emocionante al mismo tiempo. ¿Cómo será el mundo en 20 años? Nadie lo sabe con certeza, pero la tecnología seguirá siendo clave.
En las ciudades, los autos autónomos podrían cambiar el transporte para siempre. La gente joven usa aplicaciones móviles para todo, desde pedir comida hasta aprender idiomas nuevos como inglés o francés.
El cambio climático también impulsa innovaciones, como energía solar, turbinas eólicas y baterías más eficientes. Las máquinas nos ayudan a resolver problemas complejos, pero a veces generan otros nuevos que nadie anticipó.
La creatividad humana sigue siendo esencial, incluso en un mundo lleno de algoritmos y automatización. Los videojuegos son otro ejemplo de cómo la tecnología mezcla arte, ciencia y diversión.
En resumen, vivimos en una era de oportunidades y desafíos sin precedentes. La música digital ha transformado cómo escuchamos nuestras canciones favoritas.
Los deportes también se benefician de la tecnología, con análisis de datos para mejorar el rendimiento de los atletas. Viajar se ha vuelto más fácil con mapas digitales y reservas en línea.
A veces, el pasado nos enseña lecciones que aplicamos al presente, mientras miramos hacia el futuro con esperanza. La comida del futuro podría venir de impresoras 3D o granjas verticales.
El espacio exterior nos llama, y empresas como SpaceX sueñan con colonizar Marte algún día.
"""

# Crear un solo modelo con state_size=2
modelo_es = markovify.Text(corpus_es, state_size=2)

# Generar 5 oraciones con parámetros para más variedad
print("=== Oraciones generadas en español ===")
for _ in range(5):
    oracion_es = modelo_es.make_sentence(tries=100, max_overlap_ratio=0.8, max_words=20)
    if oracion_es:
        print(f"Texto generado: {oracion_es}")
    else:
        print("No se pudo generar una oración válida (intenta con un corpus más grande).")

=== Oraciones generadas en español ===
Texto generado: Otros creen que el ser humano siempre tendrá el control y que la tecnología seguirá siendo clave.
Texto generado: Algunos expertos dicen que el futuro con esperanza.
Texto generado: Los videojuegos son otro ejemplo de cómo la tecnología seguirá siendo clave.
Texto generado: Algunos expertos dicen que el ser humano siempre tendrá el control y que la tecnología mezcla arte, ciencia y diversión.
Texto generado: Viajar se ha vuelto más fácil con mapas digitales y nuevas formas de aprendizaje a distancia.


### 🔗 Explicación de por qué lo anterior funcionó mejor.

🚨 **¿Qué es max_overlap_ratio?**

El parámetro max_overlap_ratio controla el grado máximo de similitud permitido entre la oración generada y cualquier segmento del corpus original. En otras palabras, limita cuánto puede "parecerse" la oración generada a una parte exacta del texto de entrada. Esto ayuda a evitar que el modelo simplemente copie oraciones existentes y fomenta la creación de texto más original.

- Rango: Es un valor entre 0 y 1 (o entre 0% y 100% si lo piensas como porcentaje).
- Valor por defecto: Si no se especifica, markovify usa un valor predeterminado (normalmente 1, lo que permite cualquier nivel de superposición).
- Uso: Se pasa como argumento en make_sentence(max_overlap_ratio=valor).

¿Cómo funciona?
- Medición de superposición: Cuando markovify genera una oración, compara las palabras de esa oración con el corpus original. - Calcula un "ratio de superposición" basado en cuántas palabras o secuencias coinciden exactamente con alguna parte del texto de entrada.
- Límite: Si la superposición supera el valor de max_overlap_ratio, el modelo descarta esa oración e intenta generar una nueva. Esto sigue hasta que encuentra una oración que cumpla el criterio o hasta que se agoten los intentos (tries).
- Efecto: Un valor más bajo fuerza al modelo a ser más creativo, mientras que un valor más alto permite más similitud con el corpus.

---

## 🔹 Sección 3: Evaluación del Modelo – Cálculo de Perplejidad y otras métricas

Esta celda define una función para calcular la perplejidad de un modelo bigrama suavizado a partir de un texto de prueba.

In [48]:
### Primero, creemos nuevamente un modelo decente y luego lo evaluamos

# Textos de ejemplo
text_en = ("New York is a city that never sleeps. The city is vibrant and full of opportunities. "
           "New York offers endless experiences in art, culture, and nightlife.")

text_es = """La tecnología ha revolucionado la forma en que vivimos nuestras vidas cotidianas. Cada día surgen nuevas herramientas que facilitan nuestras tareas diarias y nos hacen más productivos.
Las personas están más conectadas que nunca gracias a internet y a las redes sociales como Facebook, Twitter o Instagram. Sin embargo, también hay desafíos importantes que enfrentar.
La privacidad es un tema candente en la era digital, y mucha gente teme que sus datos sean utilizados sin permiso. Algunos expertos dicen que el futuro será dominado por la inteligencia artificial y las máquinas avanzadas.
Otros creen que el ser humano siempre tendrá el control y que la tecnología solo será una herramienta en nuestras manos. La educación también se transforma con cursos en línea, plataformas digitales y nuevas formas de aprendizaje a distancia.
Todo esto nos lleva a reflexionar sobre cómo usamos estas herramientas en nuestra vida diaria. El mundo avanza rápido, y adaptarse es esencial para no quedarse atrás en este entorno cambiante.
Por ejemplo, las empresas ahora dependen de la tecnología para competir en el mercado global. Además, la inteligencia artificial está siendo utilizada en campos como la medicina, la ingeniería y el entretenimiento.
Aunque hay beneficios, también hay riesgos que debemos considerar con cuidado. El futuro es incierto, pero emocionante al mismo tiempo. ¿Cómo será el mundo en 20 años? Nadie lo sabe con certeza, pero la tecnología seguirá siendo clave.
En las ciudades, los autos autónomos podrían cambiar el transporte para siempre. La gente joven usa aplicaciones móviles para todo, desde pedir comida hasta aprender idiomas nuevos como inglés o francés.
El cambio climático también impulsa innovaciones, como energía solar, turbinas eólicas y baterías más eficientes. Las máquinas nos ayudan a resolver problemas complejos, pero a veces generan otros nuevos que nadie anticipó.
La creatividad humana sigue siendo esencial, incluso en un mundo lleno de algoritmos y automatización. Los videojuegos son otro ejemplo de cómo la tecnología mezcla arte, ciencia y diversión.
En resumen, vivimos en una era de oportunidades y desafíos sin precedentes. La música digital ha transformado cómo escuchamos nuestras canciones favoritas.
Los deportes también se benefician de la tecnología, con análisis de datos para mejorar el rendimiento de los atletas. Viajar se ha vuelto más fácil con mapas digitales y reservas en línea.
A veces, el pasado nos enseña lecciones que aplicamos al presente, mientras miramos hacia el futuro con esperanza. La comida del futuro podría venir de impresoras 3D o granjas verticales.
El espacio exterior nos llama, y empresas como SpaceX sueñan con colonizar Marte algún día."""

# Tokenización
tokens_en = tokenize_text(text_en, language='english')
tokens_es = tokenize_text(text_es, language='spanish')

# Construcción de bigramas
bigrams_en = list(bigrams(tokens_en))
bigrams_es = list(bigrams(tokens_es))

# Frecuencias de unigramas y bigramas
freq_uni_en = FreqDist(tokens_en)
freq_bi_en = FreqDist(bigrams_en)
freq_uni_es = FreqDist(tokens_es)
freq_bi_es = FreqDist(bigrams_es)

# Tamaño del vocabulario
V_en = len(set(tokens_en))
V_es = len(set(tokens_es))



# === Funciones de evaluación ===

def compute_perplexity(tokens, freq_uni, freq_bi, V):
    """
    Calcula la perplejidad de un modelo bigrama con suavizado Laplace.
    La perplejidad mide qué tan bien el modelo predice el texto: valores más bajos indican mejor predicción.

    Args:
        tokens (list): Lista de tokens del texto de prueba (ej. ['the', 'cat', 'runs']).
        freq_uni (dict): Frecuencia de unigramas del corpus (ej. {'the': 10, 'cat': 5}).
        freq_bi (dict): Frecuencia de bigramas del corpus (ej. {('the', 'cat'): 3}).
        V (int): Tamaño del vocabulario (número de palabras únicas en el corpus).

    Returns:
        float: Perplejidad del modelo.
    """
    N = len(tokens)  # Número total de tokens
    log_prob_sum = 0  # Suma de logaritmos de probabilidades

    # Iteramos sobre los bigramas del texto de prueba
    for i in range(1, N):
        bg = (tokens[i-1], tokens[i])  # Bigrama actual (palabra anterior, palabra actual)
        # Suavizado Laplace: (count(bigrama) + 1) / (count(unigrama_previo) + V)
        # Si el bigrama no existe, freq_bi[bg] es 0, pero el +1 evita probabilidad 0
        prob = (freq_bi.get(bg, 0) + 1) / (freq_uni.get(tokens[i-1], 0) + V)
        log_prob_sum += math.log(prob)  # Sumamos el logaritmo de la probabilidad

    # Perplejidad = exp(- (1/(N-1)) * suma de log(prob))
    # N-1 porque contamos bigramas (parejas), no tokens individuales
    perplexity = math.exp(-log_prob_sum / (N - 1))
    return perplexity

def compute_entropy(tokens, freq_uni, freq_bi, V):
    """
    Calcula la entropía de un modelo bigrama con suavizado Laplace.
    La entropía mide la incertidumbre promedio del modelo (en bits por palabra si usamos log base 2).
    Valores más bajos indican un modelo más predecible (menos incertidumbre).

    Args:
        tokens (list): Lista de tokens del texto de prueba.
        freq_uni (dict): Frecuencia de unigramas del corpus.
        freq_bi (dict): Frecuencia de bigramas del corpus.
        V (int): Tamaño del vocabulario.

    Returns:
        float: Entropía del modelo (en nats, usando log natural).
    """
    N = len(tokens)
    log_prob_sum = 0

    # Similar a perplejidad, calculamos la suma de -log(prob)
    for i in range(1, N):
        bg = (tokens[i-1], tokens[i])
        prob = (freq_bi.get(bg, 0) + 1) / (freq_uni.get(tokens[i-1], 0) + V)
        log_prob_sum += math.log(prob)  # Usamos log natural (base e)

    # Entropía = - (1/(N-1)) * suma de log(prob)
    entropy = -log_prob_sum / (N - 1)
    return entropy

def compute_cross_entropy(tokens, freq_uni, freq_bi, V):
    """
    Calcula la entropía cruzada de un modelo bigrama con suavizado Laplace.
    La entropía cruzada mide la diferencia entre la distribución real (texto de prueba) y la predicha por el modelo.
    Es similar a la entropía, pero conceptualmente compara con una distribución "verdadera" implícita.
    Valores más bajos indican mejor ajuste del modelo al texto de prueba.

    Args:
        tokens (list): Lista de tokens del texto de prueba.
        freq_uni (dict): Frecuencia de unigramas del corpus.
        freq_bi (dict): Frecuencia de bigramas del corpus.
        V (int): Tamaño del vocabulario.

    Returns:
        float: Entropía cruzada del modelo (en nats).
    """
    N = len(tokens)
    log_prob_sum = 0

    # Calculamos la suma de -log(prob) como en entropía
    for i in range(1, N):
        bg = (tokens[i-1], tokens[i])
        prob = (freq_bi.get(bg, 0) + 1) / (freq_uni.get(tokens[i-1], 0) + V)
        log_prob_sum += math.log(prob)

    # Entropía cruzada = - (1/(N-1)) * suma de log(prob)
    # En este caso, asumimos que el texto de prueba representa la distribución real
    cross_entropy = -log_prob_sum / (N - 1)
    return cross_entropy

# === Ejemplo de evaluación ===

# Suponiendo que estas variables ya están definidas:
# freq_uni_en/es: diccionario de frecuencias de unigramas
# freq_bi_en/es: diccionario de frecuencias de bigramas
# V_en/es: tamaño del vocabulario

texto_prueba_en = "The quick brown fox jumps over the lazy dog."
tokens_prueba_en = texto_prueba_en.split()

texto_prueba_es = "Todo esto nos lleva a reflexionar sobre cómo usamos estas herramientas en nuestra vida diaria"
tokens_prueba_es = texto_prueba_es.split()

# Evaluación en inglés
print("\n=== Evaluación del modelo en inglés ===")
perp_en = compute_perplexity(tokens_prueba_en, freq_uni_en, freq_bi_en, V_en)
entropy_en = compute_entropy(tokens_prueba_en, freq_uni_en, freq_bi_en, V_en)
cross_entropy_en = compute_cross_entropy(tokens_prueba_en, freq_uni_en, freq_bi_en, V_en)

print(f"Perplejidad: {perp_en:.4f}")
print(f"Entropía: {entropy_en:.4f} nats")
print(f"Entropía cruzada: {cross_entropy_en:.4f} nats")

# Evaluación en español
print("\n=== Evaluación del modelo en español ===")
perp_es = compute_perplexity(tokens_prueba_es, freq_uni_es, freq_bi_es, V_es)
entropy_es = compute_entropy(tokens_prueba_es, freq_uni_es, freq_bi_es, V_es)
cross_entropy_es = compute_cross_entropy(tokens_prueba_es, freq_uni_es, freq_bi_es, V_es)

print(f"Perplejidad: {perp_es:.4f}")
print(f"Entropía: {entropy_es:.4f} nats")
print(f"Entropía cruzada: {cross_entropy_es:.4f} nats")


=== Evaluación del modelo en inglés ===
Perplejidad: 14.0000
Entropía: 2.6391 nats
Entropía cruzada: 2.6391 nats

=== Evaluación del modelo en español ===
Perplejidad: 180.8195
Entropía: 5.1975 nats
Entropía cruzada: 5.1975 nats


---
## Conclusiones y Debate

En este cuaderno hemos abordado:
- Los fundamentos teóricos de los modelos probabilísticos de lenguaje.
- La construcción de modelos bigrama en inglés y español utilizando **NLTK** y **spaCy**.
- La aplicación del suavizado Laplace para estimar probabilidades y evitar el problema de la dispersión.
- La evaluación de los modelos mediante la métrica de perplejidad.

**Preguntas para el debate:**
- ¿Qué ventajas y limitaciones encuentras en los modelos n-grama para tareas de PLN?
- ¿Cómo influye el tamaño del corpus y el vocabulario en la perplejidad del modelo?
- ¿Qué otros métodos (como modelos neuronales) podrían superar algunas de estas limitaciones?